<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/wilson_ray_chi2_perp_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wilson-Ray EWPD Test (χ²⊥)

This notebook computes the covariance-weighted orthogonal deviation statistic

\[
\chi^2_{\perp}
= \hat{\mathbf C}^{T}\Sigma^{-1}\hat{\mathbf C}
-\frac{(\mathbf v^{T}\Sigma^{-1}\hat{\mathbf C})^{2}}{\mathbf v^{T}\Sigma^{-1}\mathbf v}.
\]

**Inputs required (same operator ordering):**
- `operators`: list of operator/coefficient labels
- `C_hat`: best-fit Wilson coefficient vector (length N)
- `Sigma`: covariance matrix (N×N), **or** uncertainties + correlation matrix
- `v_ray`: your ray direction vector (length N), scale irrelevant

No external frameworks are used; this is a single linear-algebra test.

## 1) Provide inputs

Choose **one** of the following input methods:
- **A.** Paste arrays directly in the next code cell.
- **B.** Load from CSV files:
  - `operators.csv` with one label per row (or a header + column `name`)
  - `C_hat.csv` with one value per row (or header + column `value`)
  - `Sigma.csv` as an N×N numeric matrix
  - `v_ray.csv` with one value per row

If you only have **uncertainties** and a **correlation matrix** `Corr`, use method C below to build `Sigma`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# A) DIRECT PASTE (edit here)
# -----------------------------
operators = None   # e.g., ["C_HWB", "C_HD", "C_ll", ...]
C_hat = None       # e.g., [0.0012, -0.0004, ...]
Sigma = None       # e.g., [[...],[...],...]
v_ray = None       # e.g., [1.0, 0.75, 1.0, ...]

# -----------------------------
# B) LOAD FROM CSV (optional)
# -----------------------------
DATA_DIR = Path(".")  # change if your files live elsewhere

def _load_vector_csv(path: Path, col_guess=("value","val","C","coeff")):
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        return df.iloc[:,0].to_numpy(dtype=float)
    for c in col_guess:
        if c in df.columns:
            return df[c].to_numpy(dtype=float)
    # fallback: first numeric column
    for c in df.columns:
        try:
            return df[c].to_numpy(dtype=float)
        except Exception:
            pass
    raise ValueError(f"Could not find numeric column in {path}")

def _load_labels_csv(path: Path, col_guess=("name","op","operator","label")):
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        return df.iloc[:,0].astype(str).tolist()
    for c in col_guess:
        if c in df.columns:
            return df[c].astype(str).tolist()
    return df.iloc[:,0].astype(str).tolist()

# Uncomment to load files if you have them:
# operators = _load_labels_csv(DATA_DIR/"operators.csv")
# C_hat = _load_vector_csv(DATA_DIR/"C_hat.csv")
# Sigma = pd.read_csv(DATA_DIR/"Sigma.csv", header=None).to_numpy(dtype=float)
# v_ray = _load_vector_csv(DATA_DIR/"v_ray.csv")

# -----------------------------
# C) BUILD Sigma from uncertainties + correlation (optional)
# -----------------------------
# If you have Corr (NxN) and sigma (N,), then Sigma = diag(sigma) @ Corr @ diag(sigma)
Corr = None   # e.g., [[1.0, 0.2, ...], ...]
sigma = None  # e.g., [0.0005, 0.0007, ...]

if Sigma is None and Corr is not None and sigma is not None:
    Corr = np.asarray(Corr, dtype=float)
    sigma = np.asarray(sigma, dtype=float).reshape(-1)
    D = np.diag(sigma)
    Sigma = D @ Corr @ D

print("Loaded?",
      "operators" if operators is not None else "",
      "C_hat" if C_hat is not None else "",
      "Sigma" if Sigma is not None else "",
      "v_ray" if v_ray is not None else "")

## 2) Validate shapes and ordering

This checks:
- lengths match N
- `Sigma` is square N×N
- `Sigma` is symmetric (within tolerance)
- optionally, positive definiteness (numerical)

In [ ]:
def validate_inputs(operators, C_hat, Sigma, v_ray, tol=1e-10):
    if operators is None or C_hat is None or Sigma is None or v_ray is None:
        raise ValueError("Missing at least one required input: operators, C_hat, Sigma, v_ray.")
    C_hat = np.asarray(C_hat, dtype=float).reshape(-1)
    v_ray = np.asarray(v_ray, dtype=float).reshape(-1)
    Sigma = np.asarray(Sigma, dtype=float)
    N = len(C_hat)
    if len(v_ray) != N:
        raise ValueError(f"Length mismatch: len(v_ray)={len(v_ray)} vs len(C_hat)={N}.")
    if len(operators) != N:
        raise ValueError(f"Length mismatch: len(operators)={len(operators)} vs len(C_hat)={N}.")
    if Sigma.shape != (N, N):
        raise ValueError(f"Sigma shape mismatch: Sigma.shape={Sigma.shape} vs (N,N)=({N},{N}).")
    if not np.allclose(Sigma, Sigma.T, atol=tol, rtol=0):
        raise ValueError("Sigma is not symmetric within tolerance. Check input.")
    # Check v not zero in metric
    # (v^T Sigma^{-1} v must be > 0)
    return operators, C_hat, Sigma, v_ray

operators, C_hat, Sigma, v_ray = validate_inputs(operators, C_hat, Sigma, v_ray)
N = len(C_hat)
print(f"N = {N} coefficients")
print("First few operators:", operators[:min(8,N)])

## 3) Compute χ²⊥

Numerically stable approach:
- solve linear systems with `Sigma` instead of explicitly computing `Sigma^{-1}`.

In [ ]:
def chi2_perp(C_hat, Sigma, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)   # N×1
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)   # N×1

    # Solve Sigma^{-1} v and Sigma^{-1} C
    Wv = np.linalg.solve(Sigma, v)
    WC = np.linalg.solve(Sigma, C)

    num = float(v.T @ WC)   # v^T Sigma^{-1} C
    den = float(v.T @ Wv)   # v^T Sigma^{-1} v
    if den <= 0:
        raise ValueError("Non-positive v^T Sigma^{-1} v; check Sigma definiteness and v_ray.")
    chi2 = float(C.T @ WC - (num*num)/den)
    return chi2, num, den

chi2, num, den = chi2_perp(C_hat, Sigma, v_ray)
print("v^T Sigma^{-1} C =", num)
print("v^T Sigma^{-1} v =", den)
print("chi2_perp        =", chi2)
print("dof              =", N-1)

## 4) Compute a p-value (optional)

This uses the chi-square distribution with dof = N−1 (Wilks regime).
If you prefer to avoid any statistical library, you can omit this step and report χ²⊥ and dof only.

In [ ]:
try:
    from scipy.stats import chi2 as chi2_dist
    p_value = 1.0 - chi2_dist.cdf(chi2, df=N-1)
    print("p-value =", p_value)
except Exception as e:
    print("scipy not available or failed; skipping p-value.")
    print("Reason:", e)

## 5) Diagnostics: fitted ray amplitude and residual vector

The covariance-weighted best-fit amplitude is
\[
\hat{\lambda} = \frac{\mathbf v^{T}\Sigma^{-1}\hat{\mathbf C}}{\mathbf v^{T}\Sigma^{-1}\mathbf v}.
\]
The orthogonal residual is \(\hat{\mathbf C}_{\perp} = \hat{\mathbf C}-\hat{\lambda}\mathbf v\).

In [ ]:
lambda_hat = num / den
C_par = lambda_hat * np.asarray(v_ray, dtype=float).reshape(-1)
C_perp = np.asarray(C_hat, dtype=float).reshape(-1) - C_par

out = pd.DataFrame({
    "operator": operators,
    "C_hat": C_hat,
    "C_parallel": C_par,
    "C_perp": C_perp,
})
out["abs_C_perp"] = np.abs(out["C_perp"])
out_sorted = out.sort_values("abs_C_perp", ascending=False)

print("lambda_hat =", lambda_hat)
display(out_sorted.head(15))